# M512 Hotel Booking Demand — Forensic EDA v2

**Purpose:** deepen v1 with source-aware variable interpretation, robustness checks, multivariable association and a formal evidence gate.

The model in this notebook is a sensitivity check only; it does not establish causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

CSV=Path("hotel_bookings.csv")
if not CSV.exists():
    import kagglehub
    folder=Path(kagglehub.dataset_download("jessemostipak/hotel-booking-demand"))
    CSV=next(iter(folder.rglob("hotel_bookings.csv")))
df=pd.read_csv(CSV)

month_num={"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,"July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df["arrival_month_num"]=df.arrival_date_month.map(month_num)
df["arrival_date"]=pd.to_datetime(dict(year=df.arrival_date_year,month=df.arrival_month_num,day=df.arrival_date_day_of_month))
df["stay_nights"]=df.stays_in_weekend_nights+df.stays_in_week_nights
df["guests"]=df.adults+df.children.fillna(0)+df.babies
df["has_prior_success"]=df.previous_bookings_not_canceled.gt(0)
df["has_special_request"]=df.total_of_special_requests.gt(0)
bins=[-1,7,30,90,180,365,np.inf]; labels=["0–7","8–30","31–90","91–180","181–365","366+"]
df["lead_band"]=pd.cut(df.lead_time,bins,labels=labels,ordered=True)
df["lead_band_str"]=df.lead_band.astype(str)

## 1. Source-aware semantics

Important source definitions:

- LeadTime is the number of days between booking entry and arrival.
- DepositType is derived from payments recorded before arrival/cancellation.
- PreviousBookingsNotCanceled is set to zero when no customer profile is associated, so zero does not cleanly mean a known customer with no successful history.
- AssignedRoomType can differ from ReservedRoomType after the original booking.
- ReservationStatus and ReservationStatusDate describe the realised outcome and are excluded as explanatory predictors.

## 2. Quality-filter sensitivity

In [ ]:
quality_filtered=df[(df.guests>0)&(df.stay_nights>0)&(df.adr>=0)&(df.adr<=df.adr.quantile(.995))].copy()

def summary(data):
    lead=data.groupby("lead_band",observed=True).is_canceled.mean()*100
    seg=data.groupby("market_segment").is_canceled.agg(["size","sum","mean"])
    return pd.Series({
        "n":len(data),
        "overall_pct":data.is_canceled.mean()*100,
        "lead_0_7_pct":lead.loc["0–7"],
        "lead_366plus_pct":lead.loc["366+"],
        "groups_pct":seg.loc["Groups","mean"]*100,
        "online_ta_pct":seg.loc["Online TA","mean"]*100
    })
display(pd.DataFrame({"Full":summary(df),"Quality-filtered":summary(quality_filtered)}).T.round(2))

## 3. Lead-time robustness across hotel, year and month

In [ ]:
display((df.pivot_table(index="lead_band",columns="hotel",values="is_canceled",aggfunc="mean",observed=True)*100).round(1))
display((df.pivot_table(index="lead_band",columns="arrival_date_year",values="is_canceled",aggfunc="mean",observed=True)*100).round(1))
display((df.pivot_table(index="lead_band",columns="arrival_date_month",values="is_canceled",aggfunc="mean",observed=True)*100).round(1))

## 4. Market segment × hotel and lead-time interactions

In [ ]:
display((df.pivot_table(index="market_segment",columns="hotel",values="is_canceled",aggfunc="mean")*100).round(1))
display(df.pivot_table(index="market_segment",columns="hotel",values="is_canceled",aggfunc="size"))

interaction=(df.groupby(["market_segment","lead_band"],observed=True).is_canceled
             .agg(bookings="size",cancellations="sum",rate="mean").reset_index())
interaction["rate"]*=100
display(interaction[interaction.bookings>=100].sort_values("cancellations",ascending=False).head(30).round(1))

## 5. Reliability signals and subgroup stability

In [ ]:
def binary_robustness(col):
    overall=df.groupby(col).is_canceled.agg(["size","mean"]); overall["mean"]*=100
    by_hotel=df.pivot_table(index=col,columns="hotel",values="is_canceled",aggfunc="mean")*100
    by_year=df.pivot_table(index=col,columns="arrival_date_year",values="is_canceled",aggfunc="mean")*100
    return overall,by_hotel,by_year

for col in ["has_prior_success","has_special_request","is_repeated_guest"]:
    print("\n###",col)
    for table in binary_robustness(col): display(table.round(1))

## 6. Multivariable robustness check — association, not causation

In [ ]:
model_df=df[df.market_segment.ne("Undefined")].copy()
formula=("is_canceled ~ C(hotel) + C(lead_band_str) + C(market_segment) + "
         "has_prior_success + has_special_request + C(customer_type) + "
         "C(arrival_month_num) + C(arrival_date_year)")
glm=smf.glm(formula,data=model_df,family=sm.families.Binomial()).fit()
ci=glm.conf_int()
odds=pd.DataFrame({
    "odds_ratio":np.exp(glm.params),
    "ci_low":np.exp(ci[0]),
    "ci_high":np.exp(ci[1]),
    "p_value":glm.pvalues
})
focus=[i for i in odds.index if any(k in i for k in ["lead_band_str","market_segment","has_prior_success","has_special_request","hotel"])]
display(odds.loc[focus].round(3))
print("Observations:",int(glm.nobs))

The adjusted model is used only to check whether the descriptive relationships disappear after adjustment for selected measured characteristics. It cannot turn observational associations into causal effects.

## 7. Exploratory commercial exposure proxy

In [ ]:
q995=df.adr.quantile(.995)
value=df[(df.stay_nights>0)&(df.adr>=0)&(df.adr<=q995)].copy()
value["booked_room_value_proxy"]=value.adr*value.stay_nights
value_seg=value.groupby("market_segment").apply(
    lambda g:pd.Series({
        "bookings":len(g),
        "cancellations":int(g.is_canceled.sum()),
        "cancelled_booked_value_proxy":g.loc[g.is_canceled.eq(1),"booked_room_value_proxy"].sum()
    }),include_groups=False)
value_seg["share_pct"]=value_seg.cancelled_booked_value_proxy/value_seg.cancelled_booked_value_proxy.sum()*100
display(value_seg.sort_values("cancelled_booked_value_proxy",ascending=False).round(1))

ADR × booked nights is only a booked-value proxy. The data do not show realised revenue loss, resale or cancellation fees, so this remains reserve evidence rather than a headline insight.

## 8. v2 evidence gate

KEEP: overall cancellation burden, robust lead-time gradient, top-three segment concentration, City-vs-Resort heterogeneity for Groups and Offline TA/TO.

KEEP WITH CAVEAT: recorded prior successful-booking history because zero also includes bookings without an associated profile.

RESERVE: special requests, distribution channel and booked-value proxy.

QUALITY ONLY: Non Refund and parking-space patterns.

REJECT AS HEADLINE: booking changes, previous cancellations and ADR bands.